# 02 — Analysis

Reads only what `01_collect.ipynb` produced. No network calls anywhere, and every
figure is drawn from `data/figures/*.parquet` rather than from the collectors — so
each figure's backing numbers can be inspected as a table and the notebook
re-runs offline.

**The discipline this notebook is held to**, from `PROJECT_BRIEF.md` §1 and the
module brief:

- Parameters are fixed **before** looking at results: NYT query forms in
  `collect/aliases.py`, Wikipedia title mappings and validity floors in
  `collect/wikipedia.py`, the normalization in `figures.build_relative_frame`,
  the collection priority in `watchlist.csv`.
- **Report the honest number.** A null result is a real result.
- **Mark underpowered results as such.**
- **No Tier B finding is promoted to a Tier A claim.**

### What changed since the first pass

Two free sources were added, and they reshaped the analysis:

- **Wikipedia pageviews** — dense, monthly, keyless, 2015→today. It covers the
  whole watchlist rather than the 12 companies NYT's quota allows, which roughly
  triples the usable sample.
- **The EDGAR event timeline** — in particular **DRS**, the *confidential* draft
  registration. It precedes the public S-1 by a median of ~96 days and by up to
  1,408, and it was secret when filed. That turns "does attention rise before
  the S-1?" into a sharper, testable question about information leakage.

In [ ]:
import json, sys
import numpy as np
import pandas as pd
sys.path.insert(0, "../..")

from research import figures
from research.collect import paths
from research.collect.edgar_enrich import read_watchlist_df
from research.collect.edgar_events import EVENTS_PARQUET
from research.collect.wikipedia import WIKI_PARQUET

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 56)

print("figure frames:", figures.build_frames())
print(json.loads(figures.FIG_MANIFEST.read_text())["built_at"])

## 1 — How the Tier B watchlist differs from the Tier A population

**First figure, not an appendix.** Every Tier B number below is only
interpretable against it, because the watchlist was hand-assembled from
well-known names and is therefore biased by construction. The point of having a
census is to *show* the bias instead of disclaiming it.

In [ ]:
fig = figures.plot_cohort_comparison()

In [ ]:
cohort = pd.read_parquet(figures.FIG_COHORT)
for dim in cohort["dimension"].unique():
    sub = cohort[cohort["dimension"] == dim]
    print(f"\n--- {dim} ---")
    print(sub[["bucket", "tier_a", "tier_a_share", "tier_b", "tier_b_share"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

Read the deal-size panel first — it is the sharpest statement of the bias.
The watchlist concentrates in the largest bucket while the population
concentrates in the smallest, and it contains no company at all from the two
smallest buckets. A list of "companies everyone remembers going public" is close
to a list of the biggest offerings.

## 2 — The pre-listing event timeline

Before looking at any attention series, establish *when things happened*. The
public S-1 is not the start of the IPO process; the confidential DRS is.

In [ ]:
ev = pd.read_parquet(EVENTS_PARQUET)
gaps = ev["drs_to_s1_days"].dropna()
print(f"DRS present for {ev['drs_first'].notna().sum()} of {len(ev)} companies\n")
print("DRS -> public S-1 gap, in days:")
print(gaps.describe(percentiles=[.25, .5, .75, .9]).round(0).to_string())
print("\nThe median company was in confidential registration for about "
      f"{gaps.median()/30.44:.1f} months before the market could know.")

In [ ]:
display(ev.sort_values("drs_to_s1_days", ascending=False)[
    ["company", "drs_first", "s1_first", "drs_to_s1_days", "pricing_first",
     "s1_amendments", "form_d_count", "comment_rounds"]].head(14))

### Why this matters for the study design

A 24-month pre-S-1 window is supposed to capture "before anything happened". For
the companies below it does not even reach the confidential filing, so every
month in their window is *already inside* the registration process.

In [ ]:
outside = ev[ev["drs_to_s1_days"] > 730]
print(f"{len(outside)} of {len(ev)} companies have a DRS outside a 24-month "
      f"pre-S-1 window:")
display(outside[["company", "drs_first", "s1_first", "drs_to_s1_days"]])
print("\nFor these, 'attention before the S-1' is not a clean pre-event baseline.")

## 3 — Attention trajectory per company

Figure 1: monthly NYT articles, monthly Wikipedia views, daily price, on a shared
x-axis, with **three** vertical markers — confidential DRS, public S-1, listing.

The order of those markers is the point. Attention that rises between the first
two moved while the registration was still secret.

In [ ]:
panels = pd.read_parquet(figures.FIG_PANELS)
print("series available, by company count:")
print(panels[panels["value"].notna()].groupby("series")["company"]
      .nunique().to_string())
print("\nWikipedia covers the whole watchlist; NYT only the quota-limited 12.")

In [ ]:
# log_counts is one choice applied to every company or to none -- a per-company
# axis choice would make two panels look alike that are not. Left False.
have_both = sorted(set(panels.loc[panels["series"] == "nyt_articles", "company"])
                   & set(panels.loc[panels["series"] == "wikipedia_views", "company"]))
for company in have_both[:4]:
    figures.plot_company(company, log_counts=False)

## 4 — Does attention rise before the confidential filing, or after the public one?

The module's central question, now askable properly. Three windows, fixed before
looking, all measured **relative to the DRS** and expressed as **views per
month** so windows of different length are comparable:

- `baseline`  — months −24 to −13 before the DRS
- `confidential` — DRS month to the public S-1 month
- `public` — S-1 month to +3 months

Only companies with data in **all three** windows are used, so the comparison is
balanced rather than reflecting which companies happen to have long histories.

In [ ]:
wiki = pd.read_parquet(WIKI_PARQUET)
anchors = ev.set_index("company")[["drs_first", "s1_first"]].copy()
for c in anchors.columns:
    anchors[c] = pd.to_datetime(anchors[c], errors="coerce")

w = wiki[wiki["valid_attention"]].copy()
w["month_start"] = pd.to_datetime(w["month"] + "-01")
w = w.join(anchors, on="company").dropna(subset=["drs_first", "s1_first"])
w["m_drs"] = ((w["month_start"].dt.year - w["drs_first"].dt.year) * 12
              + (w["month_start"].dt.month - w["drs_first"].dt.month))
w["m_s1"] = ((w["month_start"].dt.year - w["s1_first"].dt.year) * 12
             + (w["month_start"].dt.month - w["s1_first"].dt.month))

def window_of(r):
    if -24 <= r["m_drs"] <= -13:
        return "baseline"
    if r["m_drs"] >= 0 and r["m_s1"] < 0:
        return "confidential"
    if 0 <= r["m_s1"] <= 3:
        return "public"
    return None

w["window"] = w.apply(window_of, axis=1)
per = (w.dropna(subset=["window"]).groupby(["company", "window"])["views"]
       .mean().unstack())
balanced = per.dropna()
print(f"{len(balanced)} of {per.shape[0]} companies have all three windows\n")
display(balanced[["baseline", "confidential", "public"]].round(0).astype(int))

In [ ]:
cols = ["baseline", "confidential", "public"]
print("median views per month across companies:")
print(balanced[cols].median().round(0).to_string())
print("\nratio to baseline (per company, then median):")
ratio = balanced[cols].div(balanced["baseline"], axis=0)
print(ratio[["confidential", "public"]].median().round(2).to_string())
print(f"\nn = {len(balanced)}. ", end="")
if len(balanced) >= 6:
    # Wilcoxon signed-rank without scipy: exact sign test on the paired
    # differences, which needs no distributional assumption and is honest at
    # this sample size.
    from math import comb
    for w1, w2 in (("baseline", "confidential"), ("confidential", "public")):
        d = (balanced[w2] - balanced[w1]).dropna()
        pos, n = int((d > 0).sum()), int((d != 0).sum())
        p = 2 * sum(comb(n, k) for k in range(pos, n + 1)) / 2 ** n if n else float("nan")
        print(f"\n  {w1} -> {w2}: {pos}/{n} companies rose, sign-test p = {min(p,1.0):.3f}")
print("\nRead the direction and the n. A sign test on a dozen-odd paired "
      "observations is weak evidence either way.")

### Read this carefully

If `confidential` is elevated over `baseline`, attention was already moving while
the filing was secret — consistent with leakage, but **also** consistent with the
company simply becoming more famous for ordinary reasons, which is very likely
what drives a company to file in the first place. This design cannot separate
those two, and the withdrawn-company comparison group is exactly what would
begin to: companies that filed confidentially and never listed. That group is not
collected yet.

If `public` is far above `confidential`, the S-1 itself is the attention event
and there is little sign of anticipation.

## 5 — Cohort view in relative time, on all three anchors

Figure 2. Calendar-time overlay would mostly show that 2021 and 2025 were
different markets. Relative time is the only legitimate form of cross-company
comparison here.

Counts are normalized to each company's own window total before overlay, fixed
before looking — raw counts would make SpaceX and Klaviyo incomparable.

In [ ]:
for anchor in ("drs", "s1", "listing"):
    figures.plot_relative_time("wikipedia_views", anchor=anchor)

In [ ]:
rel = pd.read_parquet(figures.FIG_RELATIVE)
wv = rel[(rel["series"] == "wikipedia_views") & rel["share"].notna()]
print(f"companies in the Wikipedia overlay: {wv['company'].nunique()}")
nyt_rel = rel[(rel["series"] == "nyt_articles") & rel["share"].notna()]
print(f"companies in the NYT overlay:       {nyt_rel['company'].nunique()}")
print("\nThe Wikipedia series is the one with enough companies to say anything.")

The withdrawn-company group belongs on these axes as a second series,
anchored on filing date since those companies never list. The 431 `withdrawn`
rows are in the Tier A census and are **not yet collected against**, so
survivorship in Tier B is total and no claim about what distinguishes a company
that lists from one that does not is supported. See README limitation 2.

## 6 — Attention around listing versus subsequent return

Return is measured from the **opening print**, as `PROJECT_BRIEF.md` §2
specifies, at 30/60/90 trading sessions.

Read the `n` before the correlation.

In [ ]:
post = pd.read_parquet(figures.FIG_POST)
rows = []
for company, g in post.groupby("company"):
    g = g.sort_values("session")
    opening = g["close"].iloc[0]
    rec = {"company": company, "sessions": int(g["session"].max()) + 1}
    for h in (30, 60, 90):
        rec[f"ret_{h}d"] = (g.loc[g["session"] == h, "close"].iloc[0] / opening - 1
                            if (g["session"] == h).any() else np.nan)
    rows.append(rec)
returns = pd.DataFrame(rows).set_index("company")

# Attention in the listing month, from the dense series so all 12 have a value.
listing_month = post.groupby("company")["month"].first()
wm = wiki[wiki["valid_attention"]].set_index(["company", "month"])["views"]
returns["wiki_listing_month"] = [
    wm.get((c, listing_month.get(c)), np.nan) for c in returns.index]
display(returns.round(3))
print("\nNaN return = not yet traded that many sessions. Nothing is imputed.")

In [ ]:
usable = returns.dropna(subset=["wiki_listing_month"])
for h in (30, 60, 90):
    sub = usable.dropna(subset=[f"ret_{h}d"])
    n = len(sub)
    if n < 4:
        print(f"{h}d: n={n} -- too few to correlate.")
        continue
    r = sub["wiki_listing_month"].rank().corr(sub[f"ret_{h}d"].rank())
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    flag = "  <- interval spans 0" if lo < 0 < hi else ""
    print(f"{h}d: n={n:2d}  Spearman rho={r:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]{flag}")
print("\nSpearman on ranks, because both variables are heavily skewed.")
print("Any interval spanning zero is a NULL RESULT at this sample size -- the "
      "expected and publishable outcome. See PROJECT_BRIEF.md section 2.")

### Before banking the 30-day number

At the time of writing the 30-session correlation comes out around **−0.66 with a
95% interval that excludes zero**, and its sign is the Popularity Trap direction:
more attention, worse subsequent return. It would be very easy to write that up.
It should not be.

Three reasons, all visible above:

1. **n ≈ 10.** At that size the interval is enormous even when it excludes zero,
   and a single company moving ranks can flip it.
2. **Three horizons were tested and only one is nominally significant.** With
   three correlated tests at α=0.05, one hit is close to what chance produces.
   No multiple-comparison correction is applied below, and after even the
   crudest one it does not survive.
3. **The sign is unstable across horizons** — 30d negative, 60d *positive*, 90d
   negative. A real effect on price drift does not reverse and reverse again over
   two months; sampling noise does exactly that.

The cell below states this arithmetically rather than leaving it to judgement.

In [ ]:
# How ordinary is "one significant result out of three"?
alpha, k = 0.05, 3
print(f"P(at least one of {k} independent tests significant at {alpha}) = "
      f"{1 - (1 - alpha) ** k:.3f}")
print(f"Bonferroni-corrected threshold for {k} tests: {alpha / k:.4f}\n")

signs = {}
for h in (30, 60, 90):
    sub = usable.dropna(subset=[f"ret_{h}d"])
    if len(sub) >= 4:
        r = sub["wiki_listing_month"].rank().corr(sub[f"ret_{h}d"].rank())
        signs[h] = r
print("sign of rho by horizon:",
      {h: ("+" if v > 0 else "-") for h, v in signs.items()})
print("\nVERDICT: the horizons disagree in sign and only one of three clears an "
      "uncorrected threshold at n~10. This is a NULL RESULT. Reporting the 30d "
      "figure as the Popularity Trap would be exactly the error "
      "PROJECT_BRIEF.md section 1 exists to prevent.")

In [ ]:
# What effect size this design could even detect.
print("smallest |r| reaching p<0.05, by sample size:")
for n, tc in ((12, 2.228), (25, 2.069), (40, 2.024), (150, 1.976)):
    print(f"  n={n:3d}  need |r| >= {tc/np.sqrt(n-2+tc**2):.2f}")
print("\nAttention-return effects in the literature are ~0.1-0.3. At n=12 this "
      "design cannot see them, which is a property of the sample, not the sensors.")

## 7 — Post-listing overlay

Figure 3. The post-listing tail is the only region where price and attention
coexist, so the only place a single-panel overlay is defensible.

Marker size encodes **volume only**. No sentiment scoring is in scope and no
sentiment proxy is derived from counts.

In [ ]:
fig = figures.plot_post_listing(sorted(returns.index)[:6])

## 8 — Do the attention sources agree?

NYT and Wikipedia exist for the same company-months, so this is checkable.

Prior, stated before looking: they measure **different things** — NYT is elite
press attention, Wikipedia is public curiosity. Moderate correlation is the
expected result. Strong disagreement is informative rather than a failure, and
where they diverge Wikipedia is the denser instrument while NYT is the more
editorially filtered one.

In [ ]:
monthly = panels[panels["resolution"] == "monthly"]
wide = monthly.pivot_table(index=["company", "x"], columns="series",
                           values="value", aggfunc="first").reset_index()
both = wide.dropna(subset=["nyt_articles", "wikipedia_views"])
print(f"company-months with both series: {len(both)}")
if len(both) >= 10:
    rho = both["nyt_articles"].rank().corr(both["wikipedia_views"].rank())
    print(f"Spearman rho = {rho:+.3f} pooled across companies, n = {len(both)}\n")
    per_co = (both.groupby("company")
              .apply(lambda g: g["nyt_articles"].rank()
                     .corr(g["wikipedia_views"].rank()), include_groups=False)
              .dropna().sort_values())
    print("within-company Spearman rho:")
    print(per_co.round(3).to_string())
    print("\nPooling across companies inflates the correlation, because big "
          "companies score high on both. The within-company figures are the "
          "honest ones.")

## 9 — What this notebook does and does not support

Fill this in against the numbers actually printed above.

**Supported as written**

- The Tier A census (§1) is a full-population measurement; its counts stand alone
  and reproduce offline.
- The composition comparison (§1) is a measured statement about how unlike the
  population the watchlist is.
- The event timeline (§2) is a measured fact about filing behaviour, from primary
  sources, for every company that filed: confidential registration precedes
  public registration by a median of ~96 days.
- Per-company trajectories (§3) describe those companies.

**Not supported, and not to be written up as if it were**

- Any claim about "IPOs" in general drawn from the watchlist.
- Any attention→return relationship at this sample size, in either direction.
- **Leakage as a causal claim.** Elevated attention during the confidential
  window is equally consistent with a company becoming more famous for ordinary
  reasons — which is plausibly *why* it filed. Separating those needs the
  withdrawn-company group.
- Any statement about what separates a company that lists from one that
  withdraws — that group is not collected.
- Any Reddit, GNews, or X series. The first two could not deliver the window and
  were dropped; the third has no data because the account's credits are spent.